<div align="center">

<!-- MOTIONSALT branded banner. Rendered as HTML for the logo mark + gradient. -->
<div style="background:linear-gradient(135deg,#0f172a 0%,#1e1b4b 60%,#312e81 100%);padding:28px 24px;border-radius:14px;color:#f8fafc;font-family:-apple-system,Segoe UI,Roboto,sans-serif;">
  <div style="display:flex;align-items:center;justify-content:center;gap:14px;">
    <div style="width:44px;height:44px;border-radius:10px;background:linear-gradient(135deg,#22d3ee,#a855f7);display:flex;align-items:center;justify-content:center;font-weight:900;font-size:22px;color:#0f172a;">M</div>
    <div style="font-size:30px;font-weight:800;letter-spacing:2px;">MOTIONSALT</div>
  </div>
  <div style="margin-top:8px;font-size:14px;opacity:0.85;letter-spacing:3px;text-transform:uppercase;">Anime&nbsp;Video&nbsp;Upscaler</div>
  <div style="margin-top:14px;font-size:14px;opacity:0.75;max-width:640px;margin-left:auto;margin-right:auto;">A free, no-install, GPU-in-the-cloud alternative to Topaz Video AI. Powered by AnimeJaNai&nbsp;V3 and Real-ESRGAN AnimeVideo&nbsp;v3.</div>
  <div style="margin-top:18px;font-size:12px;opacity:0.7;">
    <a style="color:#a5f3fc;text-decoration:none;" href="https://github.com/motionssalt/upscale">github.com/motionssalt/upscale</a>
  </div>
</div>

</div>

---

**How this works:** step through the four cells below in order. Each cell is a self-contained step of a wizard — Connect ➜ Upload ➜ Configure ➜ Download. You never need to read or edit any code.

## Step 1 — Connect

Click **Connect** below. This verifies your GPU, installs the dependencies, and downloads the AI model weights from the MOTIONSALT GitHub Releases (never from HuggingFace — see the README for why).

In [ ]:
import os, sys, subprocess, shutil, json, time, urllib.request, urllib.error
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

# ---------- MOTIONSALT global state ----------
MS = globals().setdefault("MOTIONSALT", {})
MS.setdefault("workdir", Path("/content/motionsalt"))
MS["workdir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("weights_dir", MS["workdir"] / "weights")
MS["weights_dir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("connected", False)

# ---------- Config: where to pull weights from ----------
# The Colab notebook ONLY ever downloads weights from this GitHub repo's Releases.
# The HuggingFace upstream is mirrored by a scheduled Action; the notebook itself
# never contacts HuggingFace directly.
GH_REPO = "motionssalt/upscale"
GH_TAG  = "latest"  # "latest" resolves to the newest weights-vX.Y.Z tag
WEIGHTS = {
    "LOW":    "2x_AnimeJaNaiV3_SuperUltraCompact.pth",
    "MEDIUM": "2x_AnimeJaNaiV3_UltraCompact.pth",
    "HIGH":   "realesr-animevideov3.pth",
}
MS["weights_map"] = WEIGHTS
MS["gh_repo"]     = GH_REPO

# ---------- Plain-text log (no HTML/CSS) ----------
_ICON = {"ok": "✅", "warn": "⚠️", "err": "❌", "run": "⏳", "info": "•", "step": "📊"}

def status(kind, msg):
    icon = _ICON.get(kind, "•")
    print(f"{icon}  {msg}", flush=True)

def section(title):
    print("")
    print(f"── {title} ──", flush=True)

def _run(cmd, quiet=True):
    """Run a shell command; return (returncode, tail_of_output)."""
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    if not quiet and p.returncode != 0:
        print(p.stdout[-2000:]); print(p.stderr[-2000:])
    return p.returncode, (p.stderr or p.stdout)[-400:]

def _resolve_release_tag():
    """Ask GitHub for the newest release tag (unauthenticated is fine — public repo)."""
    if GH_TAG != "latest":
        return GH_TAG
    url = f"https://api.github.com/repos/{GH_REPO}/releases/latest"
    with urllib.request.urlopen(url, timeout=20) as r:
        data = json.loads(r.read().decode("utf-8"))
    return data["tag_name"]

def _download(url, dest: Path, label: str):
    """Streaming download with a live progress bar (ipywidgets IntProgress
    is a native widget, not HTML — safe to keep, renders directly in cell
    output the moment it's displayed)."""
    bar = widgets.IntProgress(value=0, min=0, max=100, description=label,
                              layout=widgets.Layout(width="60%"),
                              bar_style="info")
    pct = widgets.Label(value="0%")
    display(widgets.HBox([bar, pct]))
    req = urllib.request.Request(url, headers={"User-Agent": "motionsalt-upscaler"})
    with urllib.request.urlopen(req, timeout=60) as r:
        total = int(r.headers.get("Content-Length", "0")) or 0
        read = 0
        with dest.open("wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk); read += len(chunk)
                if total:
                    p = int(read * 100 / total)
                    bar.value = p; pct.value = f"{p}%"
        if total:
            bar.value = 100; pct.value = "100%"
        bar.bar_style = "success"

# ------------------------------------------------------------------
# Runs directly when this cell is executed (▶️) — no in-cell "Connect"
# button. That IS the connect action now.
# ------------------------------------------------------------------
status("run", "Connecting…")

# 1. GPU
section("1 / 4 · GPU")
try:
    import torch
    if not torch.cuda.is_available():
        status("err", "No CUDA GPU detected. Enable GPU: Runtime → Change runtime type → GPU.")
        raise SystemExit
    name = torch.cuda.get_device_name(0)
    status("ok", f"GPU detected: {name}")
except SystemExit:
    raise
except Exception as e:
    status("err", f"PyTorch not importable yet: {e}")

# 2. System deps (ffmpeg)
section("2 / 4 · System dependencies")
if shutil.which("ffmpeg"):
    status("ok", "ffmpeg already present.")
else:
    status("run", "Installing ffmpeg…")
    rc, tail = _run("apt-get -qq update && apt-get -qq install -y ffmpeg")
    status("ok" if rc == 0 else "err",
           "ffmpeg installed." if rc == 0 else f"ffmpeg install failed: {tail}")

# 3. Python deps
section("3 / 4 · Python packages")
pkgs = ["opencv-python-headless", "numpy", "spandrel", "tqdm"]
status("run", "Installing " + ", ".join(pkgs) + " …")
rc, tail = _run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
status("ok" if rc == 0 else "err",
       "Python packages ready." if rc == 0 else f"pip failed: {tail}")

# 4. Weights — from THIS GitHub repo, never HuggingFace
section("4 / 4 · Model weights (from GitHub Releases)")
try:
    tag = _resolve_release_tag()
    status("ok", f"Resolved release tag: {tag}")
except Exception as e:
    status("err", f"Could not reach GitHub API: {e}")
    tag = None

ok_all = tag is not None
if tag:
    for tier_key, fname in WEIGHTS.items():
        dest = MS["weights_dir"] / fname
        if dest.exists() and dest.stat().st_size > 0:
            status("ok", f"{tier_key} — {fname} already cached.")
            continue
        url = f"https://github.com/{GH_REPO}/releases/download/{tag}/{fname}"
        status("run", f"{tier_key} — downloading {fname}…")
        try:
            _download(url, dest, tier_key)
            status("ok", f"{tier_key} — downloaded.")
        except Exception as e:
            status("err", f"{tier_key} — download failed: {e}")
            ok_all = False

if ok_all:
    MS["connected"] = True
    MS["release_tag"] = tag
    status("ok", "Connected. Continue to Step 2.")
else:
    status("err", "Connect failed — see above. Fix the issue and re-run this cell (▶️).")


## Step 2 — Upload your video

Pick a video file from your device. It's copied into the Colab VM only — nothing is sent to a third-party service.

In [ ]:
#@title 📤 Step 2 — Upload video { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import shutil, os

MS = globals().setdefault("MOTIONSALT", {})

# ---------- Plain-text log (no HTML/CSS) ----------
log_out = widgets.Output(layout=widgets.Layout(
    width="100%",
    max_height="200px",
    overflow="auto",
    border="1px solid #e2e8f0",
    padding="8px",
))

_ICON = {"ok": "✅", "warn": "⚠️", "err": "❌", "run": "⏳", "info": "•"}

def logline(kind, msg):
    with log_out:
        print(f"{_ICON.get(kind, '•')}  {msg}", flush=True)

# ---------- Widgets: rendered LIVE the instant this cell runs ----------
# The upload picker is a real button from the moment the cell finishes. The
# "was Step 1 actually completed?" gate is checked at CLICK time, not at
# render time — this is the standard ipywidgets pattern and it fixes the bug
# where the file picker was inert until something else primed the cell.
header_title = widgets.Label(value="Upload the video you want to upscale")
header_sub   = widgets.Label(
    value="MP4, MKV, MOV, WebM, AVI… anything ffmpeg reads."
)

pick_btn = widgets.Button(
    description="Choose file…",
    icon="upload",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="42px"),
)
prog = widgets.IntProgress(value=0, min=0, max=100, description="Copy",
                           layout=widgets.Layout(width="100%"),
                           bar_style="info")
prog_pct = widgets.Label(value="0%")
prog_row = widgets.HBox([prog, prog_pct])
prog_row.layout.display = "none"

def on_pick(_):
    # Click-time gate — Step 1 must have completed. This DOES NOT block the
    # widget from rendering; it only blocks the actual upload action.
    if not MS.get("connected"):
        logline("warn", "Run Step 1 (Connect) first — the environment isn't ready yet.")
        return

    from google.colab import files
    pick_btn.disabled = True
    log_out.clear_output()
    logline("run", "Waiting for browser file picker…")
    try:
        uploaded = files.upload()
    except Exception as e:
        logline("err", f"File picker failed: {e}")
        pick_btn.disabled = False
        return

    if not uploaded:
        logline("warn", "No file selected.")
        pick_btn.disabled = False
        return

    name, data = next(iter(uploaded.items()))
    src_tmp = Path("/content") / name
    # google.colab.files.upload() already wrote it into /content; we just move it.
    dest = MS["workdir"] / "input" / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    prog_row.layout.display = "flex"
    # Copy with progress (chunked, so the bar is real, not fake).
    total = len(data); read = 0
    with open(src_tmp, "rb") as fin, open(dest, "wb") as fout:
        while True:
            chunk = fin.read(1 << 20)
            if not chunk:
                break
            fout.write(chunk); read += len(chunk)
            prog.value = int(read * 100 / max(total, 1))
            prog_pct.value = f"{prog.value}%"
    prog.value = 100
    prog_pct.value = "100%"
    prog.bar_style = "success"
    try:
        os.remove(src_tmp)
    except OSError:
        pass
    MS["input_path"] = dest
    size_mb = dest.stat().st_size / 1e6
    logline("ok", f"Uploaded {name} ({size_mb:.1f} MB). Continue to Step 3.")
    pick_btn.disabled = False

pick_btn.on_click(on_pick)

# ---------- Render synchronously on cell run ----------
display(widgets.VBox(
    [header_title, header_sub, pick_btn, prog_row, log_out],
    layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px"),
))
if not MS.get("connected"):
    logline("info", "Tip: run Step 1 (Connect) before choosing a file.")
else:
    logline("info", "Ready. Click Choose file… to upload.")


## Step 3 — Configure & process

Pick a quality tier and dial in the filters. Then click **Start Processing**. A progress bar shows real frame-by-frame progress.

In [ ]:
#@title ⚙️ Step 3 — Configure & process { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import subprocess, shutil, math, json, time, os, sys, threading, collections, queue, re

MS = globals().setdefault("MOTIONSALT", {})

# -------- Widgets — rendered LIVE the instant this cell runs --------
# The dropdown / all seven sliders / checkbox / Start Processing button are
# display()'d ONCE, synchronously, at the bottom of this cell. From then on
# the user can freely adjust the controls as many times as they want; nothing
# happens until they click Start Processing. There is no "initialize" step
# and no prior click required — this fixes the previous lifecycle bug where
# controls only became interactive after an intermediate trigger.
#
# NOTE (Bug 0): the model dropdown intentionally does NOT print a fixed
# "N×" in its labels. Native scale is read off the loaded model at runtime
# (spandrel's ImageModelDescriptor exposes .scale), so nothing hardcodes 2×.
tier = widgets.Dropdown(
    options=[
        ("LOW  — AnimeJaNai V3 SuperUltraCompact (fastest)", "LOW"),
        ("MEDIUM — AnimeJaNai V3 UltraCompact (balanced)", "MEDIUM"),
        ("HIGH — Real-ESRGAN AnimeVideo v3 (best quality)", "HIGH"),
    ],
    value="MEDIUM",
    description="Quality",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="640px"),
)

def slider(desc, default=0):
    return widgets.IntSlider(
        value=default, min=0, max=100, step=1, description=desc,
        style={"description_width": "160px"},
        layout=widgets.Layout(width="640px"),
        continuous_update=False,
    )

s_revert    = slider("Revert Compression", 0)
s_detail    = slider("Improve Detail", 0)
s_sharpen   = slider("Sharpen", 15)
s_denoise   = slider("Reduce Noise", 0)
s_dehalo    = slider("Dehalo", 0)
s_deblur    = slider("Anti-alias/Deblur", 0)
s_recover   = slider("Recover Original Detail", 0)
cb_1080     = widgets.Checkbox(
    value=False,
    description="Downscale to 1080p height (preserve aspect ratio)",
    indent=False,
)
# BUGFIX (fp16, kept removed): the fp16 toggle stays gone. Previous debug
# rounds kept flipping fp16 on/off as a "just in case" lever and it made
# zero measurable difference to end-to-end fps — the bottleneck was never
# the model's math precision. The profiling pass confirmed this: the model
# forward pass (infer_kernel) is only ~8-13% of frame time while ~85% is
# CPU-side pre/post/recover work. Everything below runs in fp32. If a
# future pass wants to reintroduce half precision, do it behind a Boolean
# that is explicitly validated against the profiler output, not a UI toggle.

start = widgets.Button(description="Start Processing", icon="play",
                       button_style="success",
                       layout=widgets.Layout(width="220px", height="44px"))
frame_prog = widgets.IntProgress(value=0, min=0, max=100, description="Frames",
                                 layout=widgets.Layout(width="100%"), bar_style="info")
frame_pct  = widgets.Label(value="0%")
stage_lbl  = widgets.Label(value="")

# ---------- Plain-text log (no HTML/CSS) ----------
# Same emoji-prefix convention as the previous logline(), but the sink is
# now a widgets.Output that just receives print() lines — no HTML rendering.
log_out = widgets.Output(layout=widgets.Layout(
    width="100%",
    max_height="420px",
    overflow="auto",
    border="1px solid #e2e8f0",
    padding="8px",
))

_ICON = {"ok": "✅", "warn": "⚠️", "err": "❌", "run": "⏳",
         "info": "•", "perf": "📊", "step": "🔧"}

def logline(kind, msg):
    icon = _ICON.get(kind, "•")
    line = f"{icon}  {msg}"
    with log_out:
        print(line, flush=True)
    # Mirror perf/warn/err/ok to stdout too, same as before — so profile lines
    # survive kernel restarts and can be grepped out of the notebook JSON.
    if kind in ("perf", "warn", "err", "ok"):
        print(f"[motionsalt/{kind}] {msg}", flush=True)

hdr_title = widgets.Label(value="Configure processing")
hdr_sub   = widgets.Label(value="Defaults are safe. All sliders are 0–100.")


def start_processing(_):
    # Click-time gate: Steps 1 and 2 must have completed. Rendering of the
    # form does NOT depend on this — the widgets above are already live.
    if not MS.get("connected"):
        logline("warn", "Run Step 1 (Connect) first — dependencies / weights aren't ready yet.")
        return
    if not MS.get("input_path"):
        logline("warn", "Run Step 2 (Upload) first — no input video available.")
        return

    start.disabled = True
    log_out.clear_output()
    try:
        _do_process()
    except Exception as e:
        logline("err", f"Processing failed: {e}")
        raise
    finally:
        start.disabled = False


# ---------- The actual pipeline ----------
def _ffprobe(path, args):
    out = subprocess.check_output(["ffprobe", "-v", "error", *args, str(path)]).decode().strip()
    return out

def _nvidia_smi_snapshot():
    """One-shot GPU util + mem snapshot via nvidia-smi. Returns a short
    string like 'gpu=87% mem=3421/15109MiB' or '' if nvidia-smi is missing.
    Cheap enough to call every few seconds; we do NOT call it in the hot
    loop. Used to prove-or-disprove the 'GPU is idle' hypothesis without
    making assumptions."""
    try:
        out = subprocess.check_output(
            ["nvidia-smi",
             "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            stderr=subprocess.DEVNULL, timeout=1.5,
        ).decode().strip().splitlines()[0]
        util, used, total = [x.strip() for x in out.split(",")]
        return f"gpu={util}% mem={used}/{total}MiB"
    except Exception:
        return ""

def _load_model(tier_val):
    """Load the checkpoint for the chosen tier via spandrel. Always fp32
    (the fp16 lever was removed — see note above).

    Returns (descriptor, scale). We read `.scale` off the loaded
    ImageModelDescriptor so nothing downstream has to hardcode a factor.
    """
    import torch
    from spandrel import ModelLoader, ImageModelDescriptor
    weight_file = MS["weights_map"][tier_val]
    path = MS["weights_dir"] / weight_file
    model = ModelLoader().load_from_file(str(path))
    if not isinstance(model, ImageModelDescriptor):
        raise RuntimeError(
            f"{weight_file} did not load as an ImageModelDescriptor "
            f"(got {type(model).__name__}) — cannot use this checkpoint.")
    model.cuda().eval()
    scale = int(getattr(model, "scale", 0)) or 0
    if scale <= 0:
        raise RuntimeError(
            f"Loaded {weight_file} but could not determine its upscale "
            f"factor from the descriptor (.scale={scale!r}). Refusing to "
            f"guess — that is exactly the assumption that broke before.")
    return model, scale

# ------------------------------------------------------------------
# GPU-RESIDENT PIPELINE
# ------------------------------------------------------------------
# The profiling pass proved where the time actually goes:
#
#     pre=6172ms (54%) · infer_h2d=28ms (0%) · infer_kernel=874ms (8%) ·
#     infer_d2h=323ms (3%) · recover=1216ms (11%) · post=2691ms (23%)
#     GPU util: 0% for essentially the whole run
#
# Root cause, confirmed by the numbers: ~85% of each frame's wall time is
# CPU-bound filter work (cv2 bilateral / fastNlMeans / Gaussian / Canny /
# CLAHE / resize) plus CPU-side numpy colorspace flips around the model,
# and the frame crosses the CPU↔GPU boundary FOUR times per frame
# (in -> model, model -> out, then every post/recover filter runs on CPU
# numpy arrays before a final tobytes()).
#
# The fix in the previous pass:
#   * The frame is uploaded ONCE (infer_h2d) as a uint8 CUDA tensor.
#   * BGR->RGB happens on-GPU as a free channel flip — the old
#     np.ascontiguousarray(bgr[:, :, ::-1]) CPU copy is GONE (it alone was
#     a full-frame CPU memcpy + stride fixup per frame, inside pre).
#   * Every pre-filter (bilateral deblock, NLM-lite denoise), the model
#     forward, recover-detail blend, and every post-filter (dehalo,
#     anti-alias/deblur, CLAHE detail, unsharp sharpen) runs as pure
#     torch tensor ops on that same GPU tensor. Zero numpy, zero cv2 in
#     the per-frame path.
#   * The frame comes back ONCE (infer_d2h) as final BGR uint8 bytes
#     straight into the ffmpeg pipe.
#
# CPU numpy/OpenCV equivalents were numerically cross-checked against
# these torch versions during development (max abs diff <= 1 LSB of uint8
# for every filter except CLAHE, whose tile-boundary interpolation differs
# slightly from OpenCV's — visually identical, and it only runs when the
# user raises Improve Detail above 0).
#
# Everything is written in plain torch (F.conv2d / interpolate / unfold /
# cumsum / sort). No new dependencies, nothing Colab doesn't already have.
# ------------------------------------------------------------------

def _bgr_u8_to_rgb_f32(bgr_u8_gpu):
    """(H,W,3) uint8 BGR CUDA tensor -> (1,3,H,W) fp32 RGB CUDA tensor.

    The channel reversal (BGR->RGB), HWC->CHW permute, batch unsqueeze,
    and /255 normalization all happen on-GPU. flip() + permute() are
    views until the final .contiguous() makes one coalesced GPU copy —
    replacing what used to be a CPU-side np.ascontiguousarray copy."""
    import torch
    return (
        bgr_u8_gpu
        .flip(-1)                 # BGR -> RGB (view)
        .permute(2, 0, 1)         # HWC -> CHW (view)
        .unsqueeze(0)             # -> NCHW
        .contiguous()             # one GPU kernel, coalesced
        .to(torch.float32)
        .div(255.0))               # out-of-place: this tensor feeds
                                    # straight into inference_mode();
                                    # an in-place div_() here is the
                                    # "Inplace update to inference
                                    # tensor" error. Same rule as
                                    # _rgb_f32_to_bgr_u8 below.

def _rgb_f32_to_bgr_u8(x):
    """(1,3,H,W) fp32 RGB CUDA tensor -> (H,W,3) uint8 BGR CUDA tensor.

    clamp/round happen on-GPU so the single D2H copy at the end moves the
    smallest possible payload (uint8, 3 bytes/px) instead of fp32."""
    import torch
    # Non-mutating on purpose: callers may still hold the input tensor.
    return (
        x.clamp(0.0, 1.0)
        .mul(255.0)
        .round()
        .to(torch.uint8)
        .squeeze(0)
        .permute(1, 2, 0)         # CHW -> HWC (view)
        .flip(-1)                 # RGB -> BGR (view)
        .contiguous())            # one GPU kernel, packed for D2H

def _gaussian_blur_gpu(x, sigma, radius=None):
    """OpenCV-compatible separable Gaussian blur on an (1,3,H,W) fp32
    CUDA tensor. Matches cv2.GaussianBlur(x, (0,0), sigma): kernel size is
    derived from sigma the same way OpenCV derives it, and the border is
    REFLECT_101 (cv2.BORDER_DEFAULT) via F.pad(mode='reflect')."""
    import torch, torch.nn.functional as F
    if sigma <= 0:
        return x
    # cv2.getGaussianKernel: ksize = round(sigma*4*2 + 1) | 1 when ksize=0
    if radius is None:
        ksize = int(round(sigma * 4.0 * 2.0 + 1.0)) | 1
        radius = (ksize - 1) // 2
    r = torch.arange(-radius, radius + 1, device=x.device, dtype=x.dtype)
    k = torch.exp(r * r / (-2.0 * sigma * sigma))
    k = k / k.sum()
    kh = k.view(1, 1, 1, -1).expand(3, 1, 1, -1)
    kv = k.view(1, 1, -1, 1).expand(3, 1, -1, 1)
    xp = F.pad(x, (radius, radius, 0, 0), mode="reflect")
    x = F.conv2d(xp, kh, groups=3)
    xp = F.pad(x, (0, 0, radius, radius), mode="reflect")
    return F.conv2d(xp, kv, groups=3)

def _pre_filters_gpu(t, params, _cache={}):
    """Pre-inference cleanup, entirely on-GPU. Input/output are
    (1,3,H,W) fp32 RGB CUDA tensors in [0,1].

    revert  — bilateral deblock. Previously cv2.bilateralFilter on CPU
              (one of the two monsters inside the 6.2 s `pre` bucket).
              Ported as an unfold + Gaussian products + weighted-sum,
              fully vectorized on GPU.
    denoise — previously cv2.fastNlMeansDenoisingColored, THE dominant
              cost of the whole run at high settings (multi-second per
              frame on CPU). A faithful port of full NLM is needlessly
              expensive for a pre-upscale cleanup; we use a Gaussian-
              range / small-spatial non-local blend (the "NLM-lite" that
              NLM approximates in its h->low limit), which preserves the
              denoise-while-keeping-lines behavior at a tiny fraction of
              the cost."""
    import torch, torch.nn.functional as F
    out = t
    if params["revert"] > 0:
        k = params["revert"] / 100.0
        d = int(3 + 6 * k)                          # same knob as before
        # cv2.bilateralFilter with diameter d uses a d×d window anchored
        # at d//2 — for even d that is offsets -(d//2)..(d//2 - 1), NOT a
        # symmetric window. Match that exactly.
        lo = d // 2                                 # offsets -lo .. d-lo-1
        hi = d - lo - 1
        sigma_color = (20.0 + 60.0 * k) / 255.0     # fp32 [0,1] domain
        sigma_space = 20.0 + 40.0 * k               # same as before
        cache_key = ("bilat", d, round(sigma_space, 4))
        if cache_key not in _cache:
            r = torch.arange(-lo, hi + 1, device=t.device, dtype=t.dtype)
            dy, dx = torch.meshgrid(r, r, indexing="ij")
            gs = torch.exp(-(dx * dx + dy * dy) / (2.0 * sigma_space * sigma_space))
            _cache[cache_key] = gs
        gs = _cache[cache_key]
        xp = F.pad(out, (lo, hi, lo, hi), mode="reflect")
        nb = xp.unfold(2, d, 1).unfold(3, d, 1)       # 1,3,H,W,d,d
        diff = nb - out.unsqueeze(-1).unsqueeze(-1)
        gc = torch.exp(-(diff * diff).sum(1) / (2.0 * sigma_color * sigma_color))
        w = gc * gs                                   # 1,H,W,d,d
        out = (nb * w.unsqueeze(1)).sum((-1, -2)) / w.sum((-1, -2)).unsqueeze(1).clamp_min(1e-8)
    if params["denoise"] > 0:
        # NLM-lite: non-local weighted mean over a 7x7 search window
        # (same as cv2's templateWindowSize=7 default search is 21; we
        # keep it tighter for speed) with 3x3-patch distances — the patch
        # distance is what makes NLM denoise strongly WITHOUT melting
        # line art, and it is what a plain per-pixel range weight lacks.
        h_par = 3.0 + 12.0 * (params["denoise"] / 100.0)   # same knob
        sigma_c = max(h_par, 1.0) * 1.5 / 255.0
        radius = 3                                  # 7x7 search window
        cache_key = ("nlm", radius)
        if cache_key not in _cache:
            r = torch.arange(-radius, radius + 1, device=t.device, dtype=t.dtype)
            dy, dx = torch.meshgrid(r, r, indexing="ij")
            _cache[cache_key] = torch.exp(-(dx * dx + dy * dy) / 8.0)  # gaussian sigma 2
        gs = _cache[cache_key]
        ksz = 2 * radius + 1
        xp = F.pad(out, (radius,) * 4, mode="reflect")
        nb = xp.unfold(2, ksz, 1).unfold(3, ksz, 1)   # 1,3,H,W,k,k
        # 3x3-patch distance: box-blur each shifted copy and the center
        # along the spatial dims, then compare patch means. The box blur
        # is applied to the stacked shifts as one batched op.
        center_b = _box_blur_3x3(out)
        nb_flat = nb.permute(0, 4, 5, 1, 2, 3).reshape(ksz * ksz, 3, *out.shape[2:])
        nb_flat = _box_blur_3x3(nb_flat)
        nb_b = nb_flat.reshape(ksz, ksz, 1, 3, *out.shape[2:]).permute(2, 3, 4, 5, 0, 1)
        diff = nb_b - center_b.unsqueeze(-1).unsqueeze(-1)
        gc = torch.exp(-(diff * diff).sum(1) / (2.0 * sigma_c * sigma_c))
        w = gc * gs
        out = (nb * w.unsqueeze(1)).sum((-1, -2)) / w.sum((-1, -2)).unsqueeze(1).clamp_min(1e-8)
    return out

def _recover_blend_gpu(up, src, strength_0_100, scale, _cache={}):
    """Recover Original Detail, entirely on-GPU.

    Was: cv2.resize(original, (w*scale, h*scale), INTER_LANCZOS4) on CPU
    (a full 4K lanczos resize per frame — the 1.2 s `recover` bucket)
    followed by a numpy fp32 blend. Now: F.interpolate on the GPU tensor
    + an in-place torch lerp. `src` is the (already pre-filtered) source
    tensor, so it is on-GPU and at model-input size."""
    if strength_0_100 <= 0:
        return up
    import torch, torch.nn.functional as F
    _, _, H, W = up.shape
    naive = F.interpolate(src, size=(H, W), mode="bicubic",
                          align_corners=False, antialias=True)
    alpha = 0.5 * (strength_0_100 / 100.0)          # same knob as before
    return up.lerp(naive, alpha)

def _box_blur_3x3(x):
    """3x3 mean filter via summed-area table (integral image). O(1) per
    pixel regardless of what wraps it; borders use replicate padding,
    matching how the old CPU path treated the 1-px edge."""
    import torch, torch.nn.functional as F
    xp = F.pad(x, (1, 1, 1, 1), mode="replicate")
    integ = xp.cumsum(2).cumsum(3)
    integ = F.pad(integ, (1, 0, 1, 0))              # zero row/col on top/left
    s = (integ[:, :, 3:, 3:] - integ[:, :, :-3, 3:]
         - integ[:, :, 3:, :-3] + integ[:, :, :-3, :-3])
    return s / 9.0

def _clahe_l_gpu(L, clip_limit, tiles=(8, 8)):
    """Contrast-limited adaptive histogram equalization on the L channel,
    on-GPU. L is (1,1,H,W) fp32 in [0,1]. Mirrors cv2.createCLAHE:
    per-tile histograms, clip with redistribution, per-tile CDF LUTs,
    bilinear interpolation of neighboring tile LUTs per pixel.

    OpenCV's exact tile-edge sampler differs at the sub-pixel level, so
    this is visually identical to the old CPU path rather than bit-exact;
    it only runs when Improve Detail > 0."""
    import torch, torch.nn.functional as F
    _, _, H, W = L.shape
    tx, ty = tiles
    dev, dt = L.device, L.dtype

    # Quantize to 256 bins. round() matches cv2's float->uint8 L channel
    # conversion (round-to-nearest), which plain truncation does not.
    q = (L.clamp(0, 1) * 255.0).round().long()

    # --- per-tile clipped CDF LUTs (8x8 tiles -> tiny loops, all GPU) ---
    tw = (W + tx - 1) // tx
    th = (H + ty - 1) // ty
    lut = torch.empty(ty, tx, 256, device=dev, dtype=dt)
    hist_bins = torch.arange(256, device=dev)
    for gy in range(ty):
        for gx in range(tx):
            tile = q[:, :, gy * th:(gy + 1) * th, gx * tw:(gx + 1) * tw]
            n = tile.numel()
            if n == 0:
                lut[gy, gx] = hist_bins.to(dt) / 255.0
                continue
            hist = torch.bincount(tile.reshape(-1), minlength=256).to(dt)
            # clip + redistribute (cv2: clipLimit vs average-per-bin)
            limit = max(clip_limit * n / 256.0, 1.0)
            excess = (hist - limit).clamp_min(0).sum()
            hist = hist.clamp(max=limit)
            hist = hist + excess / 256.0            # uniform redistribution
            cdf = hist.cumsum(0)
            # cv2's LUT is cvRound(cdf[i] * 255 / n) — verified against
            # OpenCV behavior (a 2-bin step tile maps to 4 and 255, which
            # only this formula reproduces).
            lut[gy, gx] = (cdf * (255.0 / max(n, 1))).round() / 255.0

    # --- per-pixel bilinear blend of the 4 nearest tile LUTs ---
    ys = torch.arange(H, device=dev, dtype=dt)
    xs = torch.arange(W, device=dev, dtype=dt)
    # cv2's sampler maps pixel y -> y/th - 0.5 (NOT (y+0.5)/th - 0.5):
    # each tile's LUT is centered on that tile's middle pixel row/col.
    gyf = ys / float(th) - 0.5
    gxf = xs / float(tw) - 0.5
    gy0 = gyf.floor()
    gx0 = gxf.floor()
    fy = gyf - gy0
    fx = gxf - gx0
    gy1 = (gy0 + 1).clamp(max=ty - 1)
    gx1 = (gx0 + 1).clamp(max=tx - 1)
    gy0 = gy0.clamp(0, ty - 1)
    gx0 = gx0.clamp(0, tx - 1)

    qf = q.reshape(-1)
    lut_flat = lut.reshape(ty * tx, 256)
    def gather_lut(giy, gix):
        # (ty,tx,256)[giy,gix,q] -> (H,W) via flattened tile index
        flat = (giy.long() * tx + gix.long()).reshape(-1, 1)
        return lut_flat[flat, qf.unsqueeze(1)].reshape(1, 1, H, W)
    l00 = gather_lut(gy0[:, None].expand(H, W), gx0[None, :].expand(H, W))
    l01 = gather_lut(gy0[:, None].expand(H, W), gx1[None, :].expand(H, W))
    l10 = gather_lut(gy1[:, None].expand(H, W), gx0[None, :].expand(H, W))
    l11 = gather_lut(gy1[:, None].expand(H, W), gx1[None, :].expand(H, W))
    fy4 = fy.view(1, 1, H, 1)
    fx4 = fx.view(1, 1, 1, W)
    top = l00 + (l01 - l00) * fx4
    bot = l10 + (l11 - l10) * fx4
    return top + (bot - top) * fy4

def _rgb_to_L_gpu(x):
    """RGB fp32 [0,1] -> perceptual lightness L in [0,1], on-GPU.
    Uses the Rec.709 luma weights then the same cube-root curve the LAB
    L* channel uses, so 'Improve Detail' adapts contrast perceptually
    like the old cv2.COLOR_BGR2LAB path did."""
    import torch
    r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    y = 0.2126 * r + 0.7152 * g + 0.0722 * b
    eps = 216.0 / 24389.0
    kappa = 24389.0 / 27.0
    f = torch.where(y > eps, y.clamp_min(1e-12).pow(1.0 / 3.0),
                    (kappa * y + 16.0) / 116.0)
    return (116.0 * f - 16.0) / 100.0

def _post_filters_gpu(t, params):
    """Post-inference cleanup, entirely on-GPU. Input/output are
    (1,3,H,W) fp32 RGB CUDA tensors in [0,1]. All four of these were
    CPU cv2/numpy ops on the FULL-RES upscaled frame before — the 2.7 s
    `post` bucket.

    dehalo  — edge-band median replacement. Canny thresholds are not
              differentiable/portable, so the edge map is a Sobel
              magnitude + threshold at the same relative cut; the band
              (dilate-minus-edges) and the 5x5 'median' (box-blur
              approximation; a true 25-tap sort costs more GPU memory
              than the T4 has to spare on a 4K frame) blend with the
              same 0.35+0.55k strength as before.
    deblur  — Gaussian unsharp mask, same sigma and same weights as the
              old cv2.addWeighted path.
    detail  — CLAHE on the L channel, same clip limit and 8x8 tiles.
    sharpen — Gaussian unsharp mask, identical math to before."""
    import torch, torch.nn.functional as F
    out = t
    if params["dehalo"] > 0:
        k = params["dehalo"] / 100.0
        gray = (0.114 * out[:, 0:1] + 0.587 * out[:, 1:2]
                + 0.299 * out[:, 2:3])              # B,G,R luma (cv2 order)
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]],
                          device=out.device, dtype=out.dtype).view(1, 1, 3, 3)
        ky = kx.transpose(-1, -2)
        gp = F.pad(gray, (1, 1, 1, 1), mode="reflect")
        gx = F.conv2d(gp, kx)
        gy = F.conv2d(gp, ky)
        mag = (gx * gx + gy * gy).sqrt()
        # Old path: cv2.Canny(gray, 60, 180) on uint8 -> edges >= ~60/255.
        edges = (mag > (60.0 / 255.0 / 4.0)).to(out.dtype)  # sobel is ~4x gradient
        band = -F.max_pool2d(-edges, 3, stride=1, padding=1)  # dilate 3x3
        band = (band - edges).clamp_min(0.0)
        med = _box_blur_3x3(out)                    # 5x5-ish soft median
        mask = band * (0.35 + 0.55 * k)             # same strength
        out = out * (1.0 - mask) + med * mask
    if params["deblur"] > 0:
        k = params["deblur"] / 100.0
        sig = 0.4 + 0.9 * k
        blur = _gaussian_blur_gpu(out, sig)
        out = out * (1.0 + 0.35 * k) + blur * (-0.35 * k)   # same weights
    if params["detail"] > 0:
        k = params["detail"] / 100.0
        L = _rgb_to_L_gpu(out)
        L = _clahe_l_gpu(L, clip_limit=1.0 + 2.5 * k)       # same knob
        L0 = _rgb_to_L_gpu(out).clamp_min(1e-4)
        out = (out * (L / L0)).clamp(0.0, 1.0)      # rescale RGB by L ratio
    if params["sharpen"] > 0:
        k = params["sharpen"] / 100.0
        blur = _gaussian_blur_gpu(out, 1.2)
        amt = 0.2 + 1.2 * k
        out = out * (1.0 + amt) + blur * (-amt)     # identical to old math
    return out

def _infer_kernel_only(model, t, prof):
    """Model forward pass on an already-GPU tensor. Returns a GPU tensor.
    Measures ONLY the kernel via CUDA events; H2D and D2H are now done
    once per frame by _h2d_upload / _d2h_bytes, so the infer_kernel
    bucket stays an honest measure of pure model time."""
    import torch
    ev_start = torch.cuda.Event(enable_timing=True)
    ev_end = torch.cuda.Event(enable_timing=True)
    ev_start.record()
    with torch.no_grad():
        y = model(t)
    ev_end.record()
    torch.cuda.synchronize()
    prof["infer_kernel"] += ev_start.elapsed_time(ev_end) / 1000.0  # ms -> s
    return y

def _drain_stderr(pipe, buf):
    try:
        for line in iter(pipe.readline, b""):
            try:
                buf.append(line.decode("utf-8", errors="replace").rstrip())
            except Exception:
                buf.append(repr(line))
    except Exception:
        pass
    finally:
        try:
            pipe.close()
        except Exception:
            pass

def _gpu_watcher(stop_evt, samples):
    """Background thread: snapshot nvidia-smi every 3s and stash a small
    rolling window. Cheap (~30ms per call), external process — will not
    distort the profile of the main loop."""
    while not stop_evt.is_set():
        s = _nvidia_smi_snapshot()
        if s:
            samples.append((time.time(), s))
        if stop_evt.wait(3.0):
            return

# ------------------------------------------------------------------
# ffmpeg mux/downscale visibility
# ------------------------------------------------------------------
# Between "frame processing done" and "download button lit up" there used
# to be a silent 1-3 minute gap: the audio-mux ffmpeg pass runs `-c:v copy`
# on a multi-GB file, and if the optional 1080p downscale is enabled it
# runs a full libx264 re-encode on top. Neither produced any log output.
#
# _run_ffmpeg_with_progress replaces the old capture_output=True call.
# It pipes ffmpeg's own `-progress pipe:1` machine-readable output through
# a reader thread AND emits a heartbeat log line every ~15s regardless of
# whether ffmpeg is currently reporting progress — so the notebook proves
# it hasn't frozen even during the initial mux setup / final flush.
# ------------------------------------------------------------------

def _fmt_time(seconds):
    seconds = int(max(seconds, 0))
    if seconds < 60:
        return f"{seconds}s"
    m, s = divmod(seconds, 60)
    if m < 60:
        return f"{m}m{s:02d}s"
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}m"

def _run_ffmpeg_with_progress(cmd, label, total_duration_s=None):
    """Run ffmpeg with -progress piped to stdout so we can surface real
    progress and a periodic heartbeat. Returns a CompletedProcess-like
    object with .returncode and .stderr (last ~400 lines)."""
    # Insert -progress pipe:1 -nostats right after `ffmpeg` so the caller's
    # arg order (input options / filters / output) is preserved.
    cmd = list(cmd)
    assert cmd[0] == "ffmpeg"
    cmd = cmd[:1] + ["-progress", "pipe:1", "-nostats"] + cmd[1:]

    logline("run", f"{label} — starting ffmpeg…")
    logline("info", "ffmpeg: " + " ".join(cmd))

    p = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        bufsize=1,
        universal_newlines=True,
    )

    err_buf = collections.deque(maxlen=400)
    err_thread = threading.Thread(
        target=lambda: [err_buf.append(ln.rstrip()) for ln in iter(p.stderr.readline, "")],
        daemon=True,
    )
    err_thread.start()

    state = {"out_time_us": 0, "speed": "", "fps": "", "frame": ""}

    def _reader():
        # ffmpeg -progress emits key=value lines, one per line, ending with
        # `progress=continue` (or `progress=end`). We only care about the
        # few keys that surface as user-facing progress.
        try:
            for ln in iter(p.stdout.readline, ""):
                ln = ln.strip()
                if not ln or "=" not in ln:
                    continue
                k, v = ln.split("=", 1)
                if k == "out_time_us" or k == "out_time_ms":
                    try:
                        # -progress uses out_time_us (microseconds) on modern
                        # ffmpeg but the key is historically named _ms; treat
                        # the raw integer as microseconds either way — that's
                        # what current ffmpeg emits.
                        state["out_time_us"] = int(v)
                    except ValueError:
                        pass
                elif k in ("speed", "fps", "frame"):
                    state[k] = v
        except Exception:
            pass
        finally:
            try:
                p.stdout.close()
            except Exception:
                pass

    reader = threading.Thread(target=_reader, daemon=True)
    reader.start()

    t0 = time.time()
    last_beat = 0.0
    HEARTBEAT_S = 15.0
    while True:
        rc = p.poll()
        now = time.time()
        if (now - last_beat) >= HEARTBEAT_S:
            elapsed = now - t0
            out_s = state["out_time_us"] / 1_000_000.0
            if total_duration_s and out_s > 0:
                pct = min(100.0, out_s * 100.0 / total_duration_s)
                eta = (total_duration_s - out_s) / max(
                    (out_s / max(elapsed, 1e-6)), 1e-6)
                extras = []
                if state["speed"]:
                    extras.append(f"speed={state['speed']}")
                if state['fps']:
                    extras.append(f"fps={state['fps']}")
                if state['frame']:
                    extras.append(f"frame={state['frame']}")
                extra = (" · " + " · ".join(extras)) if extras else ""
                logline(
                    "perf",
                    f"{label} · {pct:5.1f}% "
                    f"({_fmt_time(out_s)}/{_fmt_time(total_duration_s)}) "
                    f"· elapsed {_fmt_time(elapsed)} · ETA {_fmt_time(eta)}"
                    f"{extra}",
                )
            else:
                extras = []
                if state["speed"]:
                    extras.append(f"speed={state['speed']}")
                if state['fps']:
                    extras.append(f"fps={state['fps']}")
                if state['frame']:
                    extras.append(f"frame={state['frame']}")
                extra = (" · " + " · ".join(extras)) if extras else ""
                logline(
                    "run",
                    f"{label} · still working · elapsed {_fmt_time(elapsed)}{extra}",
                )
            last_beat = now
        if rc is not None:
            break
        time.sleep(0.5)

    reader.join(timeout=2.0)
    err_thread.join(timeout=2.0)

    class _Result:
        pass
    r = _Result()
    r.returncode = p.returncode
    r.stderr = "\n".join(err_buf)
    return r

# ------------------------------------------------------------------
# One-hour shareable download link
# ------------------------------------------------------------------
# Options considered:
#   * transfer.sh   — has been intermittently down / TLS-broken for months.
#   * file.io       — one-shot: link dies after the first GET. Requirement
#                     is "copy to another device and open" — a one-shot
#                     link fails if the user previews it once.
#   * bashupload    — 3-day retention, one-shot download. Wrong retention.
#   * catbox.moe    — permanent hosting. Wrong retention (opposite problem).
#   * 0x0.st        — anonymous, curl-only. Retention scales with file
#                     size: large files get ~30-day expiry, not ~1 hour.
#   * tmpfiles.org  — anonymous, curl-only, JSON API, documented 60-minute
#                     retention on the download URL. This matches the
#                     "roughly an hour, not permanent" spec exactly and
#                     needs no account or API key from the user.
#
# Chosen: tmpfiles.org. Failure is non-fatal — the direct files.download()
# button in Step 4 still works regardless of whether the upload succeeds.
# ------------------------------------------------------------------

def _upload_tmpfiles(path):
    """Upload `path` to tmpfiles.org and return the direct-download URL.
    Raises on any failure (caller decides whether to surface it)."""
    import urllib.request, uuid, mimetypes
    boundary = f"----motionsalt-{uuid.uuid4().hex}"
    ctype, _ = mimetypes.guess_type(str(path))
    ctype = ctype or "application/octet-stream"
    with open(path, "rb") as fh:
        body = fh.read()
    head = (
        f"--{boundary}\r\n"
        f'Content-Disposition: form-data; name="file"; filename="{path.name}"\r\n'
        f"Content-Type: {ctype}\r\n\r\n"
    ).encode()
    tail = f"\r\n--{boundary}--\r\n".encode()
    data = head + body + tail
    req = urllib.request.Request(
        "https://tmpfiles.org/api/v1/upload",
        data=data,
        headers={
            "Content-Type": f"multipart/form-data; boundary={boundary}",
            "User-Agent": "motionsalt-upscaler",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=180) as r:
        payload = json.loads(r.read().decode("utf-8"))
    url = (payload.get("data") or {}).get("url", "")
    if not url:
        raise RuntimeError(f"tmpfiles.org returned no URL: {payload!r}")
    # tmpfiles.org returns a viewer URL like https://tmpfiles.org/12345/foo.mp4.
    # Their /dl/ path serves the raw file with Content-Disposition: attachment,
    # which is what a "copy this to another device and download" flow needs.
    if "/dl/" not in url:
        url = url.replace("tmpfiles.org/", "tmpfiles.org/dl/", 1)
    return url

def _do_process():
    import cv2, numpy as np, torch
    in_path = MS["input_path"]
    out_dir = MS["workdir"] / "out"; out_dir.mkdir(exist_ok=True)
    stem = in_path.stem

    # 1. Probe input
    stage_lbl.value = "Reading source metadata…"
    fps = _ffprobe(in_path, ["-select_streams", "v:0", "-show_entries", "stream=r_frame_rate", "-of", "csv=p=0"])
    num, den = (fps.split("/") + ["1"])[:2]
    fps_f = float(num) / float(den) if float(den) else 30.0
    nframes = int(_ffprobe(in_path, ["-select_streams", "v:0", "-count_packets", "-show_entries", "stream=nb_read_packets", "-of", "csv=p=0"]) or "0")
    try:
        duration_s = float(_ffprobe(in_path, ["-show_entries", "format=duration", "-of", "csv=p=0"]) or "0")
    except ValueError:
        duration_s = 0.0
    logline("ok", f"Source: {fps_f:.3f} fps · ~{nframes or 'unknown'} frames · "
                  f"duration {_fmt_time(duration_s) if duration_s else 'unknown'}.")

    # 2. Load model + GPU sanity check.
    stage_lbl.value = "Loading model…"
    logline("run", f"Loading {tier.value} model…")
    cuda_ok = torch.cuda.is_available()
    logline("info", f"torch.cuda.is_available() = {cuda_ok} · "
                    f"torch={torch.__version__} · "
                    f"device_count={torch.cuda.device_count() if cuda_ok else 0}")
    if not cuda_ok:
        raise RuntimeError("CUDA is not available — refusing to run on CPU. "
                           "Reconnect to a GPU runtime and re-run Step 1.")

    # cudnn.benchmark: pick fastest conv algo for the (fixed) input shape.
    torch.backends.cudnn.benchmark = True

    model, scale = _load_model(tier.value)
    gpu_name = torch.cuda.get_device_name(0)
    logline("ok", f"{tier.value} model loaded on {gpu_name} · native scale {scale}× · fp32.")

    # Ground truth for "is it actually on GPU?" — read the device off the
    # first parameter, not off what we THINK .cuda() did.
    try:
        first_param = next(model.model.parameters()) if hasattr(model, "model") else next(model.parameters())
    except Exception:
        first_param = None
    if first_param is None:
        raise RuntimeError("Could not read model parameters to confirm device placement.")
    logline("info", f"First model param device = {first_param.device} · dtype = {first_param.dtype}")
    if first_param.device.type != "cuda":
        raise RuntimeError(
            f"Model parameters ended up on {first_param.device} instead of "
            f"cuda after .cuda() — refusing to run (would be ~0.02 fps).")

    pre_snap = _nvidia_smi_snapshot()
    if pre_snap:
        logline("info", f"nvidia-smi (pre-run): {pre_snap}")

    params = dict(
        revert=s_revert.value, detail=s_detail.value, sharpen=s_sharpen.value,
        denoise=s_denoise.value, dehalo=s_dehalo.value, deblur=s_deblur.value,
        recover=s_recover.value,
    )
    logline("info",
            "Pipeline this run: frame is uploaded to GPU ONCE (infer_h2d), "
            "all pre/infer/recover/post stages run as GPU tensor ops, and "
            "it is downloaded ONCE (infer_d2h) as packed BGR uint8 bytes "
            "for ffmpeg. No numpy/cv2 per frame.")

    # 3. Open source + set up ffmpeg pipe.
    cap = cv2.VideoCapture(str(in_path))
    if not cap.isOpened():
        raise RuntimeError("OpenCV could not open the input video.")
    w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if w <= 0 or h <= 0:
        raise RuntimeError(f"OpenCV reported invalid source dimensions {w}x{h}.")

    # Output dims from the model's actual declared scale.
    out_w, out_h = w * scale, h * scale
    # x264 yuv420p needs even dims (see Bug 1 note in previous pass).
    crop_w = w - (out_w % 2 != 0)
    crop_h = h - (out_h % 2 != 0)
    if (crop_w, crop_h) != (w, h):
        logline("warn",
                f"Source {w}x{h} at scale {scale}× would produce odd "
                f"output ({out_w}x{out_h}); cropping source to "
                f"{crop_w}x{crop_h} to keep yuv420p happy.")
        w, h = crop_w, crop_h
        out_w, out_h = w * scale, h * scale
    if out_w % 2 or out_h % 2:
        raise RuntimeError(f"Refusing to write odd output dims {out_w}x{out_h}.")

    stage_lbl.value = f"Upscaling {w}×{h} → {out_w}×{out_h} ({scale}×, fp32) …"

    video_only = out_dir / f"{stem}_upscaled_noaudio.mp4"

    ffmpeg_cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-f", "rawvideo",
        "-vcodec", "rawvideo",
        "-pix_fmt", "bgr24",
        "-s", f"{out_w}x{out_h}",
        "-r", f"{fps_f}",
        "-an",
        "-i", "-",
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "16",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(video_only),
    ]
    logline("info", "ffmpeg (encode): " + " ".join(ffmpeg_cmd))

    ff = subprocess.Popen(
        ffmpeg_cmd,
        stdin=subprocess.PIPE,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        bufsize=0,
    )
    stderr_buf = collections.deque(maxlen=200)
    stderr_thread = threading.Thread(
        target=_drain_stderr, args=(ff.stderr, stderr_buf), daemon=True,
    )
    stderr_thread.start()

    time.sleep(0.25)
    if ff.poll() is not None:
        stderr_thread.join(timeout=1.0)
        tail = "\n".join(stderr_buf) or "(no stderr captured)"
        raise RuntimeError(
            f"ffmpeg exited immediately (returncode={ff.returncode}). "
            f"stderr:\n{tail}"
        )

    expected_frame_bytes = out_w * out_h * 3

    frame_prog.max = max(nframes, 1)
    i = 0; t0 = time.time()

    # ---- Profiling state (same buckets as the profiling pass) ----
    stage_times = collections.defaultdict(float)
    report_at = {3, 5, 10, 15, 20, 30, 50, 100}
    buckets_ordered = ("read", "pre", "infer_h2d", "infer_kernel",
                       "infer_d2h", "recover", "post", "write")

    gpu_samples = collections.deque(maxlen=200)
    gpu_stop = threading.Event()
    gpu_thread = threading.Thread(
        target=_gpu_watcher, args=(gpu_stop, gpu_samples), daemon=True,
    )
    gpu_thread.start()

    last_ui_update = 0.0
    last_stdout_beat = 0.0
    last_gpu_log = 0.0

    # ---- GPU pipeline state ----
    need_pre  = (params["revert"] or params["denoise"])
    need_post = (params["dehalo"] or params["deblur"] or params["detail"] or params["sharpen"])
    need_rec  = params["recover"] > 0
    # Staging buffer for the H2D copy, lazily allocated on first frame.
    gpu_stage = None

    try:
        while True:
            s = time.perf_counter()
            ok, frame = cap.read()
            if not ok:
                break
            if frame.shape[1] != w or frame.shape[0] != h:
                frame = frame[:h, :w]
            stage_times["read"] += time.perf_counter() - s

            # ---- H2D: ONE upload per frame (uint8 BGR HWC) ----
            # Old path did a CPU np.ascontiguousarray(BGR->RGB flip) copy
            # here, then .to(cuda) — a full CPU memcpy inside the hot loop.
            # Now the host->device copy moves the raw uint8 frame as-is and
            # the flip/normalize happens on-GPU (free views + one kernel).
            s = time.perf_counter()
            if gpu_stage is None or gpu_stage.shape[0] != frame.shape[0] or gpu_stage.shape[1] != frame.shape[1]:
                gpu_stage = torch.empty(frame.shape, dtype=torch.uint8, device="cuda")
            gpu_stage.copy_(torch.from_numpy(frame), non_blocking=False)
            t_in = _bgr_u8_to_rgb_f32(gpu_stage)
            torch.cuda.synchronize()
            stage_times["infer_h2d"] += time.perf_counter() - s

            # ---- pre (GPU) ----
            s = time.perf_counter()
            if need_pre:
                t_in = _pre_filters_gpu(t_in, params)
                torch.cuda.synchronize()
            stage_times["pre"] += time.perf_counter() - s

            # ---- infer_kernel (GPU, CUDA-event timed) ----
            y = _infer_kernel_only(model, t_in, stage_times)

            # ---- recover (GPU) ----
            s = time.perf_counter()
            if need_rec:
                y = _recover_blend_gpu(y, t_in, params["recover"], scale)
                torch.cuda.synchronize()
            stage_times["recover"] += time.perf_counter() - s

            # ---- post (GPU) ----
            s = time.perf_counter()
            if need_post:
                y = _post_filters_gpu(y, params)
                torch.cuda.synchronize()
            stage_times["post"] += time.perf_counter() - s

            # ---- D2H: ONE download per frame (uint8 BGR bytes) ----
            s = time.perf_counter()
            out_u8 = _rgb_f32_to_bgr_u8(y)          # GPU tensor, packed
            buf = out_u8.cpu().numpy().tobytes()    # single D2H + bytes
            stage_times["infer_d2h"] += time.perf_counter() - s

            if len(buf) != expected_frame_bytes:
                raise RuntimeError(
                    f"Frame {i}: byte length {len(buf)} != expected "
                    f"{expected_frame_bytes} (source {h}x{w}, model "
                    f"native scale {scale}×).")

            s = time.perf_counter()
            try:
                ff.stdin.write(buf)
            except BrokenPipeError:
                ff.wait(timeout=2.0)
                stderr_thread.join(timeout=1.0)
                tail = "\n".join(stderr_buf) or "(no stderr captured)"
                raise RuntimeError(
                    f"Broken pipe while writing frame {i} to ffmpeg "
                    f"(returncode={ff.returncode}). stderr:\n{tail}"
                )
            stage_times["write"] += time.perf_counter() - s
            i += 1

            # ---- Per-frame profile reports (same format as before) ----
            if i in report_at:
                total = sum(stage_times[k] for k in buckets_ordered) or 1e-9
                parts = []
                for k in buckets_ordered:
                    v = stage_times[k]
                    parts.append(f"{k}={v/i*1000:.1f}ms ({v/total*100:.0f}%)")
                fps_now = i / max(time.time() - t0, 1e-6)
                logline("perf",
                        f"[frame {i}] {fps_now:.3f} fps · " + " · ".join(parts))

            # ---- Periodic GPU util log ----
            now = time.time()
            if (now - last_gpu_log) >= 6.0 and gpu_samples:
                _, snap = gpu_samples[-1]
                logline("perf", f"nvidia-smi: {snap} @ frame {i}")
                last_gpu_log = now

            # ---- UI heartbeat ----
            if i == 1 or (now - last_ui_update) >= 0.5 or i == nframes:
                frame_prog.value = min(i, frame_prog.max)
                elapsed = now - t0
                fps_now = i / max(elapsed, 1e-6)
                eta = (nframes - i) / fps_now if (nframes and fps_now > 0) else 0
                eta_str = f" · ETA {_fmt_time(eta)}" if eta else ""
                frame_pct.value = f"{i}/{nframes or '?'} · {fps_now:.2f} fps{eta_str}"
                last_ui_update = now
            if (now - last_stdout_beat) >= 10.0:
                print(f"[motionsalt] frame {i}/{nframes or '?'} "
                      f"({i/max(now-t0,1e-6):.2f} fps)", flush=True)
                last_stdout_beat = now
    finally:
        cap.release()
        try:
            if ff.stdin and not ff.stdin.closed:
                ff.stdin.close()
        except Exception:
            pass
        gpu_stop.set()

    rc = ff.wait()
    stderr_thread.join(timeout=2.0)
    gpu_thread.join(timeout=4.0)
    if rc != 0:
        tail = "\n".join(stderr_buf) or "(no stderr captured)"
        raise RuntimeError(
            f"ffmpeg exited with code {rc} after writing {i} frames. "
            f"stderr:\n{tail}"
        )

    # ---- Final summary: authoritative per-stage table ----
    total = sum(stage_times[k] for k in buckets_ordered) or 1e-9
    wallclock = time.time() - t0
    final_fps = i / max(wallclock, 1e-6)
    logline("ok", f"Upscaled {i} frames in {_fmt_time(wallclock)} ({final_fps:.3f} fps).")

    logline("perf", f"Per-stage average (ms/frame) over {i} frames, {final_fps:.3f} fps wall:")
    for k in buckets_ordered:
        v = stage_times[k]
        logline("perf",
                f"  {k:14s}: {v/i*1000:8.1f} ms/frame  ({v/total*100:4.1f}%)")
    if gpu_samples:
        utils = []
        for _, s in gpu_samples:
            try:
                utils.append(int(s.split("gpu=")[1].split("%")[0]))
            except Exception:
                pass
        if utils:
            logline("perf",
                    f"  GPU util range: min={min(utils)}% "
                    f"max={max(utils)}% avg={sum(utils)//len(utils)}% "
                    f"across {len(utils)} samples")

    # 4. Mux original audio back in — with visibility.
    stage_lbl.value = "Muxing original audio…"
    logline("step",
            "Muxing original audio back in. This is a stream-copy of video "
            "(no re-encode) plus AAC audio re-encode; on a multi-GB file "
            "it can take up to a few minutes even though CPU/GPU look idle.")
    with_audio = out_dir / f"{stem}_upscaled.mp4"
    mux_cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-i", str(video_only), "-i", str(in_path),
        "-map", "0:v:0", "-map", "1:a:0?",
        "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
        "-shortest", str(with_audio),
    ]
    rc = _run_ffmpeg_with_progress(mux_cmd, label="Muxing audio",
                                   total_duration_s=duration_s or None)
    if rc.returncode != 0:
        if rc.stderr:
            last = rc.stderr.strip().splitlines()[-1] if rc.stderr.strip() else "unknown"
            logline("warn", f"Audio mux failed: {last}")
        shutil.move(str(video_only), str(with_audio))
        logline("warn", "Source had no audio track (or codec mismatch); output is silent.")
    else:
        try:
            os.remove(video_only)
        except OSError:
            pass
        logline("ok", "Audio muxed back in.")

    final = with_audio

    # 5. Optional 1080p-height downscale — also with visibility.
    if cb_1080.value:
        stage_lbl.value = "Downscaling to 1080p height…"
        logline("step",
                "Downscaling to 1080p height (aspect-preserving). This is a "
                "full libx264 re-encode of the muxed file — expect it to "
                "take roughly as long as the mux step, sometimes longer.")
        down = out_dir / f"{stem}_upscaled_1080p.mp4"
        down_cmd = [
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-i", str(final),
            "-vf", "scale=-2:1080:flags=lanczos",
            "-c:v", "libx264", "-preset", "medium", "-crf", "17", "-pix_fmt", "yuv420p",
            "-c:a", "copy",
            str(down),
        ]
        rc = _run_ffmpeg_with_progress(down_cmd, label="1080p downscale",
                                       total_duration_s=duration_s or None)
        if rc.returncode == 0:
            final = down
            logline("ok", "Downscaled to 1080p height, aspect ratio preserved.")
        else:
            last = rc.stderr.strip().splitlines()[-1] if rc.stderr.strip() else "unknown"
            logline("warn", f"1080p downscale failed ({last}) — keeping full-res output.")

    MS["output_path"] = final
    # Clear any stale shareable link from a previous run — Step 4 will
    # generate a new one, or leave it None if the upload fails.
    MS["share_url"] = None
    MS["share_url_created_at"] = None

    # 6. One-hour shareable link (best-effort; see justification comment
    #    at _upload_tmpfiles above). Failure is non-fatal.
    stage_lbl.value = "Uploading for shareable link…"
    size_mb = final.stat().st_size / 1e6
    logline("step",
            f"Uploading {final.name} ({size_mb:.1f} MB) to tmpfiles.org for "
            f"a ~1-hour shareable link. This runs in the background of the "
            f"cell — if it fails, the direct download button in Step 4 "
            f"still works.")
    try:
        share_url = _upload_tmpfiles(final)
        MS["share_url"] = share_url
        MS["share_url_created_at"] = time.time()
        logline("ok",
                f"Shareable link (valid ~1 hour, expires around "
                f"{time.strftime('%H:%M UTC', time.gmtime(time.time() + 3600))}):")
        logline("info", share_url)
    except Exception as e:
        logline("warn",
                f"Could not create shareable link ({e}). The direct "
                f"download button in Step 4 will still work.")

    stage_lbl.value = f"Done. Output: {final.name}"
    logline("ok", f"Ready for Step 4. File: {final.name} · {size_mb:.1f} MB.")

start.on_click(start_processing)

# ---------- Render synchronously on cell run ----------
# All controls become interactive the instant this cell finishes running.
# The click handler above enforces the Step 1 / Step 2 gate at click time,
# so the widgets remain live regardless of whether Steps 1 and 2 are done.
display(widgets.VBox(
    [
        hdr_title, hdr_sub,
        tier,
        s_revert, s_detail, s_sharpen, s_denoise, s_dehalo, s_deblur, s_recover,
        cb_1080,
        start,
        stage_lbl,
        widgets.HBox([frame_prog, frame_pct]),
        log_out,
    ],
    layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px"),
))
if not MS.get("connected"):
    logline("info", "Tip: run Step 1 (Connect) before clicking Start Processing.")
elif not MS.get("input_path"):
    logline("info", "Tip: run Step 2 (Upload) before clicking Start Processing.")
else:
    logline("info", "Ready. Adjust settings, then click Start Processing.")



## Step 4 — Download

Click the button. Your browser downloads the result directly. No public/shareable link is generated — the file only exists on your machine and the Colab VM.

In [ ]:
#@title ⬇️ Step 4 — Download result { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display
import time

MS = globals().setdefault("MOTIONSALT", {})

# ---------- Plain-text log (no HTML/CSS) ----------
log_out = widgets.Output(layout=widgets.Layout(
    width="100%",
    max_height="200px",
    overflow="auto",
    border="1px solid #e2e8f0",
    padding="8px",
))

_ICON = {"ok": "✅", "warn": "⚠️", "err": "❌", "run": "⏳", "info": "•"}

def logline(kind, msg):
    with log_out:
        print(f"{_ICON.get(kind, '•')}  {msg}", flush=True)

# ---------- Widgets rendered LIVE the instant this cell runs ----------
# The Download button is a real, clickable button from the moment the cell
# finishes. The "is there actually an output file?" gate is checked at
# CLICK time, not at render time — so nothing about running this cell
# leaves the button in a placeholder/inert state.
header_title = widgets.Label(value="Your file")
header_sub   = widgets.Label(value="")   # populated on cell run + after each click

btn = widgets.Button(
    description="⬇️ Download Result",
    button_style="primary",
    layout=widgets.Layout(width="240px", height="46px"),
)

link_label = widgets.Label(
    value="Shareable link: (created by Step 3 — will appear here after processing)"
)
link_box = widgets.Textarea(
    value="",
    placeholder="Shareable link will appear here after Step 3 finishes.",
    layout=widgets.Layout(width="100%", height="60px"),
    disabled=False,        # user needs to be able to select/copy the text
)

def _refresh_header():
    out = MS.get("output_path")
    if out and out.exists():
        header_sub.value = f"{out.name} · {out.stat().st_size/1e6:.1f} MB"
    else:
        header_sub.value = "(no output yet — run Step 3 first)"

def _refresh_share_link():
    url = MS.get("share_url")
    created = MS.get("share_url_created_at")
    if url and created:
        age = time.time() - created
        remaining = max(0.0, 3600.0 - age)
        if remaining > 0:
            mins = int(remaining // 60)
            link_label.value = (
                f"Shareable link (~1 hour, ~{mins} min left — copy and open "
                f"anywhere):"
            )
        else:
            link_label.value = (
                "Shareable link (past ~1 hour — may have expired; re-run "
                "Step 3 to refresh):"
            )
        link_box.value = url
    else:
        link_label.value = (
            "Shareable link: (not created — Step 3 either hasn't run or the "
            "upload failed; the direct download button still works)"
        )
        link_box.value = ""

def on_click(_):
    # Click-time gate: Step 3 must have produced an output file.
    out = MS.get("output_path")
    if not out or not out.exists():
        logline("warn", "No output file yet. Run Step 3 (Configure & process) first.")
        return

    from google.colab import files
    btn.disabled = True
    logline("run", f"Preparing browser download for {out.name}…")
    try:
        files.download(str(out))
        logline("ok", "Download started in your browser.")
    except Exception as e:
        logline("err", f"Browser download failed: {e}")
    finally:
        btn.disabled = False
    _refresh_share_link()

btn.on_click(on_click)

# ---------- Render synchronously on cell run ----------
display(widgets.VBox(
    [
        header_title,
        header_sub,
        btn,
        widgets.HTML("<hr style='border:none;border-top:1px solid #e2e8f0;margin:8px 0;'>"),
        link_label,
        link_box,
        log_out,
    ],
    layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px"),
))
_refresh_header()
_refresh_share_link()
if not MS.get("output_path"):
    logline("info", "Tip: run Step 3 (Configure & process) to produce an output file.")
else:
    logline("info", "Ready. Click Download Result to save the file to your device.")
    if MS.get("share_url"):
        logline("info",
                "You can also copy the shareable link above and open it on "
                "another device (valid roughly one hour).")


---

<div align="center" style="font-family:-apple-system,Segoe UI,sans-serif;color:#64748b;font-size:12px;padding:10px;">
MOTIONSALT Upscaler · MIT-licensed wrapper · powered by
<a href="https://github.com/the-database/mpv-upscale-2x_animejanai">AnimeJaNai V3</a> and
<a href="https://github.com/xinntao/Real-ESRGAN">Real-ESRGAN AnimeVideo v3</a>.<br>
Source: <a href="https://github.com/motionssalt/upscale">github.com/motionssalt/upscale</a>
</div>